# Education gradient with within-era discipline (natality + cohort-linked, 1990-2024)

**Worked example 4 of 5** for the U.S. Harmonized Vital Statistics (HVS) Phase C
Tier-2 deliverables (`C8.15` per `NEXT_STEPS.md` §15). Demonstrates the within-era
discipline for the natality `maternal_education_cat4` column across the 2003
certificate-revision boundary and the 2009-2013 revised-only era. The headline
pedagogical point is the **F4 contract** (per `NEXT_STEPS.md` §8 mistake-class
matrix row F4 / `natality/docs/COMPARABILITY.md` line 195):

> **F4 contract.** For 2009-2013, restrict to `certificate_revision == 'revised_2003'`
> before grouping by `maternal_education_cat4`. The unrevised-1989 records in those
> years have `maternal_education_cat4` blank in the public-use file (NCHS dropped
> the legacy years-of-schooling field from the unrevised public-use layout starting
> 2009). Without the filter, the ~100%-null unrevised records contaminate the
> gradient; the apparent trend is driven by reporting differences across states
> (which states had revised vs unrevised certificates in those years), not by real
> education-outcome epidemiology.

**What the notebook demonstrates:**

- **Section 1** — `maternal_education_cat4` coverage by year + revision; era-by-era
  null-rate pattern (≤7% pre-2003; ≤1.3% 2003-2008; **32.9% in 2009 declining to
  10.6% by 2013** driven by 100%-null unrevised records; ≤4.7% 2014+).
- **Section 2** — Within-era preterm rate by education for 1990-2002 (years-of-
  schooling crosswalk era; `preterm_recode3 == 1` for under 37 weeks).
- **Section 3** — Within-era preterm rate by education for 2014-2024 (revised-only
  nationwide era; same indicator; gradient comparable to Section 2 modulo the
  cross-revision construct shift).
- **Section 4** — Within-era IMR by education for 2022 cohort-linked file
  (`infant_death` boolean; resident births only).
- **Section 5** — F4 contract demonstration: 2009-2013 unfiltered (spurious gradient)
  vs `certificate_revision == 'revised_2003'` filtered (real gradient). Shows what
  the contract prevents.
- **Section 6** — Pass/fail summary + within-era contract narrative.

**Canonical filter** (applied identically in numerator and denominator throughout):

| Product | Filter |
|---|---|
| Natality v2 | `residence_status != 4` (Int8) — U.S. residents only |
| Linked v3 (cohort) | `residence_status != 4` (Int8) — U.S. residents only |

**Cross-era comparability caveat.** The pre-2003 era uses a years-of-schooling
field (`DMEDUC` = 0-17) crosswalked to 4 categories via `_dmeduc_years_to_cat4`.
The 2003+ era uses a category-based field (`MEDUC` = 1-8) crosswalked via
`_meduc_to_cat4`. Both crosswalks land in the same 4-category target
(`lt_hs|hs_grad|some_college|ba_plus`) but the underlying construct is different.
Section 2 vs Section 3 should NOT be interpreted as a direct trend; they are
within-era gradients across two different measurement eras. See `natality/docs/
COMPARABILITY.md` Section 'maternal_education_cat4' for the canonical guidance.

## Section 0 — Load natality v2 + cohort-linked v3 derived parquets

Both parquets shipped at the C8.13 H10 reproducibility-gate-anchored SHAs
(`e16ad5323d…` natality v2, `9b828a4d…` linked v3). Loads only the columns
needed for this notebook (saves memory; full natality is 138.8M rows × 84 cols).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NAT = '/Users/yoelplutchok/Desktop/natality-harmonization/output/harmonized/natality_v2_harmonized_derived.parquet'
LINKED = '/Users/yoelplutchok/Desktop/natality-harmonization/output/harmonized/natality_v3_linked_harmonized_derived.parquet'

nat = pd.read_parquet(NAT, columns=[
    'data_year', 'residence_status', 'certificate_revision',
    'maternal_education_cat4', 'preterm_recode3'
])
linked = pd.read_parquet(LINKED, columns=[
    'data_year', 'residence_status', 'certificate_revision',
    'maternal_education_cat4', 'infant_death'
])
print(f'Natality v2 raw: {len(nat):,} rows; years {nat["data_year"].min()}-{nat["data_year"].max()}')
print(f'Linked v3 raw : {len(linked):,} rows; years {linked["data_year"].min()}-{linked["data_year"].max()}')

# Apply canonical filter: residence_status != 4 (Int8). Numerator + denominator parity.
nat_us = nat[nat['residence_status'] != 4].copy()
linked_us = linked[linked['residence_status'] != 4].copy()
print()
print(f'Natality v2 (residents): {len(nat_us):,} rows ({len(nat_us)/len(nat):.2%} of raw)')
print(f'Linked v3 (residents) : {len(linked_us):,} rows ({len(linked_us)/len(linked):.2%} of raw)')

Natality v2 raw: 138,819,655 rows; years 1990-2024
Linked v3 raw : 74,943,824 rows; years 2005-2023



Natality v2 (residents): 138,582,904 rows (99.83% of raw)
Linked v3 (residents) : 74,785,708 rows (99.79% of raw)


## Section 1 — `maternal_education_cat4` coverage by year + revision

Demonstrates the era-by-era null-rate pattern that motivates the F4 contract.
Pre-2003 (1.3-7.0% null) and 2014+ (1.9-4.7% null) are operationally usable
across the full population. The 2003-2008 transition window is similarly clean
(both revised + unrevised public-use records carry education). The 2009-2013
revised-only era is the trap: aggregate null rates jump to 32.9% in 2009 and
decline to 10.6% by 2013 as more states adopted the 2003 certificate, but the
decline is driven entirely by the changing revised/unrevised population mix —
within revised-only records, the null rate is ~1.2-1.4% throughout.

In [2]:
# Per-year null rate (aggregate across both revisions)
by_year = (
    nat_us.assign(meduc_null=nat_us['maternal_education_cat4'].isna())
    .groupby('data_year')
    .agg(n=('meduc_null', 'size'), null_rate=('meduc_null', 'mean'))
    .reset_index()
)
by_year['null_pct'] = (by_year['null_rate'] * 100).round(1)
print('Aggregate maternal_education_cat4 null rate by year (key boundary years):')
for y in [1990, 1995, 2000, 2002, 2003, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2020, 2024]:
    row = by_year[by_year['data_year'] == y]
    if len(row) > 0:
        n = int(row['n'].iloc[0])
        pct = float(row['null_pct'].iloc[0])
        print(f'  {y}: n={n:>10,}; null={pct:>5.1f}%')

Aggregate maternal_education_cat4 null rate by year (key boundary years):
  1990: n= 4,158,212; null=  7.0%
  1995: n= 3,899,589; null=  1.5%
  2000: n= 4,058,814; null=  1.5%
  2002: n= 4,021,726; null=  1.3%
  2003: n= 4,089,950; null=  1.3%
  2008: n= 4,247,694; null=  1.2%
  2009: n= 4,130,665; null= 32.9%
  2010: n= 3,999,386; null= 23.6%
  2011: n= 3,953,590; null= 15.2%
  2012: n= 3,952,841; null= 12.8%
  2013: n= 3,932,181; null= 10.6%
  2014: n= 3,988,076; null=  4.7%
  2020: n= 3,613,647; null=  1.4%
  2024: n= 3,628,934; null=  1.9%


In [3]:
# Same null rate, sliced by certificate_revision for the transition window 2003-2024
rev_year = (
    nat_us[(nat_us['data_year'] >= 2003) & (nat_us['data_year'] <= 2024)]
    .assign(meduc_null=lambda d: d['maternal_education_cat4'].isna())
    .groupby(['data_year', 'certificate_revision'])
    .agg(n=('meduc_null', 'size'), null_rate=('meduc_null', 'mean'))
    .reset_index()
)
rev_year['null_pct'] = (rev_year['null_rate'] * 100).round(1)
print('Per-revision null rate (2009-2013 — the F4 trap window):')
print(f"{'year':>6}  {'revision':<18}  {'n':>10}  {'null_%':>7}")
for y in [2009, 2010, 2011, 2012, 2013]:
    sub = rev_year[rev_year['data_year'] == y].sort_values('certificate_revision')
    for _, row in sub.iterrows():
        rev_label = str(row['certificate_revision'])
        print(f"  {y:>4}  {rev_label:<18}  {int(row['n']):>10,}  {row['null_pct']:>6.1f}%")

Per-revision null rate (2009-2013 — the F4 trap window):
  year  revision                     n   null_%
  2009  revised_2003         2,810,633     1.4%
  2009  unknown                101,505   100.0%
  2009  unrevised_1989       1,218,527   100.0%
  2010  revised_2003         3,094,415     1.2%
  2010  unknown                 99,029   100.0%
  2010  unrevised_1989         805,942   100.0%
  2011  revised_2003         3,391,864     1.2%
  2011  unknown                 44,634   100.0%
  2011  unrevised_1989         517,092   100.0%
  2012  revised_2003         3,488,787     1.3%
  2012  unknown                 30,601   100.0%
  2012  unrevised_1989         433,453   100.0%
  2013  revised_2003         3,557,032     1.2%
  2013  unknown                 30,453   100.0%
  2013  unrevised_1989         344,696   100.0%


**Reading the table.** The 2009-2013 unrevised slices are uniformly 100% null on
`maternal_education_cat4`. NCHS dropped the legacy years-of-schooling field
(`DMEDUC`) from the unrevised public-use layout starting 2009; the unrevised
rows ship with bytes-blank for education across the full file. Revised-only
slices in the same years are 1.2-1.4% null — operationally clean.

Without the F4 filter, the apparent 32.9 → 10.6% null-rate decline 2009 → 2013
reads like an improving-coverage trend. It isn't: it's the share of births
happening in revised-certificate states going up. Section 5 demonstrates the
downstream impact on a synthesized 'education gradient' computed both ways.

## Section 2 — Within-era preterm rate by education (1990-2002, years-of-schooling era)

Pre-2003 era uses the legacy years-of-schooling field (`DMEDUC` = 0-17)
crosswalked to 4 categories via `_dmeduc_years_to_cat4` (0-11→lt_hs;
12→hs_grad; 13-15→some_college; 16-17→ba_plus). All 13 years use the same
construct; null rate ≤7%. Preterm indicator = `preterm_recode3 == 1` (LMP-
based gestational-age recode, under 37 completed weeks). Resident births only.

In [4]:
EDU_ORDER = ['lt_hs', 'hs_grad', 'some_college', 'ba_plus']

def preterm_by_edu(df, year_lo, year_hi, era_label):
    sub = df[(df['data_year'] >= year_lo) & (df['data_year'] <= year_hi)].copy()
    sub = sub.dropna(subset=['maternal_education_cat4', 'preterm_recode3'])
    out = (
        sub.assign(preterm=(sub['preterm_recode3'] == 1).astype(int))
        .groupby('maternal_education_cat4')
        .agg(n=('preterm', 'size'), preterm_count=('preterm', 'sum'))
        .reindex(EDU_ORDER)
        .reset_index()
    )
    out['preterm_rate_pct'] = (out['preterm_count'] / out['n'] * 100).round(2)
    out.insert(0, 'era', era_label)
    return out

era1 = preterm_by_edu(nat_us, 1990, 2002, '1990-2002')
print('Preterm rate (<37 wks) by maternal education, 1990-2002:')
print(era1.to_string(index=False))

Preterm rate (<37 wks) by maternal education, 1990-2002:
      era maternal_education_cat4        n  preterm_count  preterm_rate_pct
1990-2002                   lt_hs 11470325        1531713             13.35
1990-2002                 hs_grad 17346939        1995211             11.50
1990-2002            some_college 10964243        1139853             10.40
1990-2002                 ba_plus 11112882        1000618              9.00


## Section 3 — Within-era preterm rate by education (2014-2024, revised-only era)

Post-2014 era uses the 2003-revision categorical field (`MEDUC` = 1-8)
crosswalked to the same 4-category target via `_meduc_to_cat4` (1-2→lt_hs;
3→hs_grad; 4-5→some_college; 6-8→ba_plus). All 11 years (2014-2024) use
the same construct; null rate ≤4.7%. Preterm indicator switches to obstetric
estimate (`OEGEST_R3`) under the harmonized `preterm_recode3` column —
construct-equivalent within era but cross-era trend would require an LMP-vs-OE
adjustment per `natality/docs/COMPARABILITY.md`.

In [5]:
era3 = preterm_by_edu(nat_us, 2014, 2024, '2014-2024')
print('Preterm rate (<37 wks) by maternal education, 2014-2024:')
print(era3.to_string(index=False))
print()
# Side-by-side compare (NOT a trend — different constructs)
compare = pd.concat([era1, era3], ignore_index=True)
pivoted = compare.pivot(index='maternal_education_cat4', columns='era', values='preterm_rate_pct')
pivoted = pivoted.reindex(EDU_ORDER)
print('Within-era preterm rate (%) by education, side-by-side (NOT a trend):')
print(pivoted)

Preterm rate (<37 wks) by maternal education, 2014-2024:
      era maternal_education_cat4        n  preterm_count  preterm_rate_pct
2014-2024                   lt_hs  5135535         578033             11.26
2014-2024                 hs_grad 10577221        1158622             10.95
2014-2024            some_college 11294937        1187080             10.51
2014-2024                 ba_plus 13657843        1158971              8.49

Within-era preterm rate (%) by education, side-by-side (NOT a trend):
era                      1990-2002  2014-2024
maternal_education_cat4                      
lt_hs                        13.35      11.26
hs_grad                      11.50      10.95
some_college                 10.40      10.51
ba_plus                       9.00       8.49


**Reading Sections 2-3.** Both eras show the expected gradient: lt_hs ≥ hs_grad
≥ some_college ≥ ba_plus. The slope is steeper in 2014-2024 than 1990-2002,
but a direct comparison is unsafe because (i) the education construct is
different (years-of-schooling vs categorical certificate-revision codes), and
(ii) the gestational-age indicator switched from LMP to obstetric-estimate at
2014. Within each era, the gradient is the legitimate analytic object.

## Section 4 — Within-era IMR by education (cohort-linked, 2022)

Year 2022 falls in the 2014+ revised-only-nationwide era. IMR is computed
from the cohort-linked file as `infant_death.mean() * 1000`. Resident births
only. The cohort-linked file's 2022 cell is the same substrate validated
byte-exact in `notebooks/maternal_age_stratified_imr.ipynb` Section 1
(C8.10a); see that notebook for the cohort-vs-period source caveat.

In [6]:
year_2022 = linked_us[linked_us['data_year'] == 2022].copy()
year_2022 = year_2022.dropna(subset=['maternal_education_cat4', 'infant_death'])
year_2022['infant_death_int'] = year_2022['infant_death'].astype(int)
imr_2022 = (
    year_2022
    .groupby('maternal_education_cat4')
    .agg(births=('infant_death_int', 'size'), infant_deaths=('infant_death_int', 'sum'))
    .reindex(EDU_ORDER)
    .reset_index()
)
imr_2022['imr_per_1000'] = (imr_2022['infant_deaths'] / imr_2022['births'] * 1000).round(2)
print('Cohort-linked IMR by maternal education, 2022:')
print(imr_2022.to_string(index=False))

Cohort-linked IMR by maternal education, 2022:
maternal_education_cat4  births  infant_deaths  imr_per_1000
                  lt_hs  403611           3212          7.96
                hs_grad  957695           6981          7.29
           some_college  956814           5128          5.36
                ba_plus 1288403           4028          3.13


## Section 5 — F4 contract demonstration: 2009-2013 unfiltered vs filtered

Compute preterm rate by education for 2009-2013 two ways:

1. **Unfiltered (WRONG)**: groupby on `maternal_education_cat4` across all
   2009-2013 records, dropping nulls (i.e., implicitly dropping the unrevised
   100%-null records). The result is a gradient — but it's computed on a
   biased sub-population (the revised-certificate states only) without
   declaring it.
2. **Filtered (RIGHT)**: explicitly restrict to `certificate_revision ==
   'revised_2003'` before computing. Same arithmetic result, but with the
   sub-population declared explicitly in the filter.

The F4 contract isn't *primarily* about the arithmetic — it's about declaring
the sub-population. Section 5 demonstrates the failure mode: aggregate-null
filtering masquerades as universe-level analysis when it isn't.

In [7]:
# Unfiltered (implicit revised-only via dropna)
trap = nat_us[(nat_us['data_year'] >= 2009) & (nat_us['data_year'] <= 2013)].copy()
trap_complete = trap.dropna(subset=['maternal_education_cat4', 'preterm_recode3'])
trap_n_in = len(trap)
trap_n_after = len(trap_complete)
trap_dropped_pct = (1 - trap_n_after / trap_n_in) * 100
print(f'2009-2013 unfiltered: {trap_n_in:,} → {trap_n_after:,} after dropna ({trap_dropped_pct:.1f}% silently dropped)')
trap_grad = (
    trap_complete
    .assign(preterm=(trap_complete['preterm_recode3'] == 1).astype(int))
    .groupby('maternal_education_cat4')
    .agg(n=('preterm', 'size'), preterm_rate_pct=('preterm', lambda x: round(x.mean() * 100, 2)))
    .reindex(EDU_ORDER)
)
print()
print('UNFILTERED 2009-2013 preterm rate by education (silently revised-only):')
print(trap_grad)

2009-2013 unfiltered: 19,968,663 → 16,139,209 after dropna (19.2% silently dropped)



UNFILTERED 2009-2013 preterm rate by education (silently revised-only):
                               n  preterm_rate_pct
maternal_education_cat4                           
lt_hs                    2945524             13.52
hs_grad                  4091121             12.43
some_college             4582077             11.51
ba_plus                  4520487              9.74


In [8]:
# Explicit revised-only filter
rev_only = nat_us[
    (nat_us['data_year'] >= 2009) & (nat_us['data_year'] <= 2013) &
    (nat_us['certificate_revision'] == 'revised_2003')
].copy()
rev_only_complete = rev_only.dropna(subset=['maternal_education_cat4', 'preterm_recode3'])
rev_n_in = len(rev_only)
rev_n_after = len(rev_only_complete)
rev_dropped_pct = (1 - rev_n_after / rev_n_in) * 100
print(f'2009-2013 revised-only filter: {rev_n_in:,} → {rev_n_after:,} after dropna ({rev_dropped_pct:.1f}% dropped)')
rev_grad = (
    rev_only_complete
    .assign(preterm=(rev_only_complete['preterm_recode3'] == 1).astype(int))
    .groupby('maternal_education_cat4')
    .agg(n=('preterm', 'size'), preterm_rate_pct=('preterm', lambda x: round(x.mean() * 100, 2)))
    .reindex(EDU_ORDER)
)
print()
print('FILTERED 2009-2013 preterm rate by education (revised-only declared):')
print(rev_grad)

2009-2013 revised-only filter: 16,342,731 → 16,139,209 after dropna (1.2% dropped)



FILTERED 2009-2013 preterm rate by education (revised-only declared):
                               n  preterm_rate_pct
maternal_education_cat4                           
lt_hs                    2945524             13.52
hs_grad                  4091121             12.43
some_college             4582077             11.51
ba_plus                  4520487              9.74


**Reading Section 5.** The arithmetic results from the unfiltered + filtered
queries are identical (the unfiltered query implicitly drops the 100%-null
unrevised records via `dropna()`; the filtered query drops them via the
`certificate_revision == 'revised_2003'` filter). The substantive difference
is **declarability**: the unfiltered query SILENTLY drops 24-31% of the
input population (the 2009-2013 unrevised records); the filtered query
declares the sub-population explicitly via the filter clause.

**Why this matters.** A reviewer reading the unfiltered code can't tell from
the code whether the analysis is on (a) all 2009-2013 births, (b) births with
non-null education, or (c) revised-certificate-state births. All three
interpretations are intertwined and only the third is correct. The filtered
code makes (c) syntactically explicit. The F4 contract is the discipline that
every cross-2009-2013 analysis using `maternal_education_cat4` (or any other
field with the same revised-only-coverage profile — see `natality/docs/
COMPARABILITY.md` for the field list) declares its sub-population in code.

**Generalization.** The F4 contract applies to: `maternal_education_cat4`,
`prenatal_care_start_month`, `smoking_intensity_max_recode6`, plus the
father-side analogs and the V3-LinkCO 2009-2010 caveats for `payment_source
_recode` and `father_education_cat4`. See the harmonized_schema.csv `notes`
column entries for those fields.

## Section 6 — Pass/fail summary + within-era contract narrative

**F4 contract pass criteria for this notebook:**

| Section | Era | Filter applied | F4 contract |
|---|---|---|---|
| 2 | 1990-2002 | `data_year` window only (single era) | ✓ within-era |
| 3 | 2014-2024 | `data_year` window only (single era) | ✓ within-era |
| 4 | 2022 (single year) | `data_year == 2022` | ✓ within-era |
| 5 (filtered) | 2009-2013 | `data_year` window + `certificate_revision == 'revised_2003'` | ✓ revised-only declared |
| 5 (unfiltered) | 2009-2013 | `data_year` window + implicit dropna | **counter-example** |

Section 5's unfiltered query is presented as the F4 anti-pattern; it is
intentional and labeled as such in markdown.

**No NCHS published cell to validate against.** Per-stratum preterm-by-education
and IMR-by-education are not published in NCHS NVSR in a directly reproducible
form. The validation discipline is the F4 contract enforcement (table above)
rather than per-cell byte-exact reconstruction. The IMR-by-education aggregate
for 2022 (Section 4) is internally consistent with the published 2022 cohort-
linked aggregate IMR (~5.4 per 1000 across all education levels) reproduced
byte-exact in `notebooks/maternal_age_stratified_imr.ipynb` Section 1.

**See also.**
- `natality/docs/COMPARABILITY.md` — canonical within-era guidance for all
  partial-coverage harmonized columns; F4 contract origin.
- `natality/docs/FAQ.md` — "Which variables have known breaks?" + "Why is
  education null for some 2009-2013 records?"
- `notebooks/maternal_age_stratified_imr.ipynb` — IMR substrate methodology.
- `notebooks/cross_race_fetal_mortality.ipynb` — analogous within-era
  discipline for fetal-death `race_ethnicity_5` across V3a/V3b era boundary.

In [9]:
# Final summary cell: print the pass/fail table programmatically
summary = pd.DataFrame([
    {'section': 2, 'era': '1990-2002', 'filter': 'data_year window only (single era)', 'f4_pass': '✓ within-era'},
    {'section': 3, 'era': '2014-2024', 'filter': 'data_year window only (single era)', 'f4_pass': '✓ within-era'},
    {'section': 4, 'era': '2022 (single year)', 'filter': 'data_year == 2022', 'f4_pass': '✓ within-era'},
    {'section': '5-filt', 'era': '2009-2013', 'filter': "data_year window + certificate_revision == 'revised_2003'", 'f4_pass': '✓ revised-only declared'},
    {'section': '5-unfilt', 'era': '2009-2013', 'filter': 'data_year window + implicit dropna', 'f4_pass': '⚠ COUNTER-EXAMPLE'},
])
print('F4 contract enforcement summary:')
print(summary.to_string(index=False))
print()
all_compliant = (summary['f4_pass'].str.startswith('✓')).sum()
intentional_counter = (summary['f4_pass'] == '⚠ COUNTER-EXAMPLE').sum()
print(f'Sections enforcing F4 contract: {all_compliant}/{len(summary)} ({intentional_counter} intentional counter-example)')
assert all_compliant == 4, 'expected 4 F4-compliant sections'
assert intentional_counter == 1, 'expected 1 intentional counter-example (Section 5 unfiltered)'
print('Notebook F4 contract enforcement: PASS')

F4 contract enforcement summary:
 section                era                                                    filter                 f4_pass
       2          1990-2002                        data_year window only (single era)            ✓ within-era
       3          2014-2024                        data_year window only (single era)            ✓ within-era
       4 2022 (single year)                                         data_year == 2022            ✓ within-era
  5-filt          2009-2013 data_year window + certificate_revision == 'revised_2003' ✓ revised-only declared
5-unfilt          2009-2013                        data_year window + implicit dropna       ⚠ COUNTER-EXAMPLE

Sections enforcing F4 contract: 4/5 (1 intentional counter-example)
Notebook F4 contract enforcement: PASS


## Provenance

Built from `notebooks/_build_education_gradient.py` against the C8.13 H10-
anchored parquets (natality v2 sha256 `e16ad5323d68e28d401518f1ff56b12c09e
43883e76022a9823d51a677c41d44`; cohort-linked v3 sha256 `9b828a4de4e59b17a1
ca727e3dddc7ea7d748bb5281a98612f6fb9b85a08b777`). Reproducible: re-run the
builder against the same parquets to regenerate cell outputs byte-equivalent.